In [ ]:
# The best thing to do first is to fix up and tidy the vanilla llm agent, this needs to work with fireworks AI. I guess the goal is to train all the agents fairly and then prepare an evaluation script with Sams evaluation dataset and benchmark their reward performance on them for N lots of 256 steps.

In [ ]:
from primaite.agents.aegis.gllm import GLLM
from primaite.agents.aegis.modules.openai import OpenAIClient
from primaite.agents.git_agent import GITAgent
from torch.utils.data import DataLoader
from primaite.agents.llm.observation import network_connectivity_desc, ObservedState
import logging

logging.disable(logging.CRITICAL)

In [ ]:
gllm = GLLM()
openai = OpenAIClient(openai_api_key="")

In [ ]:
questions = [
    "What is the total number of nodes in the network?",
    "Describe the network",
    "What connects to CLIENT_1?",
    "What connects to the management console?",
    "What is connected to SWITCH_2?",
    "Which nodes are compromised if any?",
    "What is the hardware state of SWITCH_1?",
    "What is the software state of WEB_SERVER?",
    "Explain the network architecture, and what connects to what",
    "How many nodes connect to the management console?",
    "How many switches are there?",
    "How many unique node types are there?",
]

In [ ]:
agent._env.active_nodes

In [ ]:
import random

random.randint(a=0, b=100)

In [ ]:
# Mock a primaite graph for development
agent = GITAgent(
    training_config_path="../agents/training_configs/git.yaml",
    lay_down_config_path="../data/laydown_configs/lay_down_config_6_data_manipulation.yaml",
)
obs = agent._env.reset()
data = agent.create_graph(obs)
observed_state = ObservedState.from_env(agent._env)
network_desc = network_connectivity_desc(observed_state.network)
obs_view_full = ObservedState.from_env(agent._env).format()
# network_states = obs_view_full(agent.env_history[-1])

In [ ]:
import os
import pickle as pkl

if "openai_responses.pkl" not in os.listdir("./"):
    openai_responses = []
    openai_prompts = gllm.build_prompts(
        questions=questions, network_desc=network_desc + "\n" + obs_view_full, model="openai"
    )
    for prompt in openai_prompts:
        openai_responses.append(openai.generate(prompt=prompt))

    with open("openai_responses.pkl", "wb") as file:
        pkl.dump(openai_responses, file)
else:
    with open("openai_responses.pkl", "rb") as file:
        openai_responses = pkl.load(file)

In [ ]:
from primaite.agents.aegis.data import GLLMDataset, collate_fn

dataset = GLLMDataset(
    graphs=[data for _ in range(len(questions))],
    questions=[question for question in questions],
    gt_answers=[response for response in openai_responses],
    llm=gllm.llm,
)

In [ ]:
dataloader = DataLoader(dataset, batch_size=12, shuffle=True, collate_fn=collate_fn)

In [ ]:
from primaite.agents.aegis.gllm import train_loop

gllm_responses = train_loop(
    model=gllm, dataloader=dataloader, network_desc=network_desc, n_epochs=2000, loss_fn="crossentropy"
)